# Semantic Segmentation — Воспроизводимое обучение

**Модели:** FCN, U-Net, DeepLabV3+, Attention U-Net, SegFormer, U-Net PT, Attention U-Net PT  
**Датасет:** ISPRS Vaihingen (6 классов, 16 изображений)  
**Устройство:** Apple MPS / NVIDIA CUDA / CPU (автоопределение)

## Порядок запуска
1. Ячейка 1 — клонирование репозитория
2. Ячейка 2 — установка зависимостей
3. Ячейка 3 — определение устройства
4. Ячейка 4 — скачивание данных (Kaggle API)
5. Ячейка 5 — предобработка (тайлинг 512x512)
6. Ячейка 6 — проверка датасета
7. Ячейка 7 — обучение всех моделей
8. Ячейка 8 — оценка на тестовой выборке
9. Ячейка 9 — финальная таблица и проверка гипотез
10. Ячейка 10 — графики кривых обучения
11. Ячейка 11 — инференс на одном изображении

In [ ]:
# Ячейка 1: Клонирование репозитория
import os, sys

REPO_URL    = 'https://github.com/safffka/diploma.git'
DIPLOMA_DIR = os.path.expanduser('~/diploma')

if not os.path.exists(DIPLOMA_DIR):
    os.system(f'git clone {REPO_URL} {DIPLOMA_DIR}')
    print(f'Клонировано: {DIPLOMA_DIR}')
else:
    os.system(f'cd {DIPLOMA_DIR} && git pull')
    print(f'Обновлено: {DIPLOMA_DIR}')

os.chdir(DIPLOMA_DIR)
sys.path.insert(0, DIPLOMA_DIR)
print(f'Рабочая папка: {os.getcwd()}')

In [ ]:
# Ячейка 2: Установка зависимостей
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Зависимости установлены')

In [ ]:
# Ячейка 3: Определение устройства
import torch, sys
print(f'Python: {sys.version.split()[0]}  PyTorch: {torch.__version__}')

if torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('Apple MPS')
elif torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
else:
    DEVICE = 'cpu'
    print('CPU')
print(f'DEVICE = "{DEVICE}"')

In [ ]:
# Ячейка 4: Скачивание датасета ISPRS Vaihingen
# Пропусти если data/processed/vaihingen/ уже содержит тайлы
#
# Для скачивания нужен ~/.kaggle/kaggle.json
# 1. https://www.kaggle.com/settings -> API -> Create New Token
# 2. cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

import os
PROCESSED = 'data/processed/vaihingen/train/images'
RAW = 'data/raw/ISPRS_semantic_labeling_Vaihingen/top'

if os.path.exists(PROCESSED) and len(os.listdir(PROCESSED)) > 0:
    print('Тайлы уже есть:')
    for split in ['train','val','test']:
        n = len(os.listdir(f'data/processed/vaihingen/{split}/images'))
        print(f'  {split}: {n} тайлов')
elif os.path.exists(RAW):
    print('Сырые данные есть — запустите ячейку 5')
else:
    kaggle_cfg = os.path.expanduser('~/.kaggle/kaggle.json')
    if not os.path.exists(kaggle_cfg):
        print('ОШИБКА: ~/.kaggle/kaggle.json не найден')
    else:
        os.makedirs('data/raw', exist_ok=True)
        print('Скачиваем датасет (~14 GB)...')
        os.system('kaggle datasets download -d bkfateam/potsdamvaihingen -p data/raw/')
        os.system('cd data/raw && unzip -q potsdamvaihingen.zip')
        print('Датасет распакован')

In [ ]:
# Ячейка 5: Предобработка (тайлинг 512x512)
# Пропусти если data/processed/vaihingen/ уже есть
import subprocess, sys, os
env = {**os.environ, 'PYTHONPATH': DIPLOMA_DIR}
result = subprocess.run(
    [sys.executable, 'scripts/preprocess_real.py',
     '--raw_dir', 'data/raw/ISPRS_semantic_labeling_Vaihingen',
     '--out_dir', 'data/processed/vaihingen'],
    text=True, cwd=DIPLOMA_DIR, env=env
)
print(result.stdout)
if result.returncode != 0:
    print('ОШИБКА:', result.stderr[-1000:])

In [ ]:
# Ячейка 6: Проверка датасета
from src.data.dataset import ISPRSDataset, get_dataset_info, ISPRS_NUM_CLASSES, IN_CHANNELS

info = get_dataset_info('vaihingen')
print(f'num_classes = {info["num_classes"]}')
print(f'class_names = {info["class_names"]}')
print(f'pixel_mean  = {[round(x,4) for x in info["pixel_mean"]]}')

train_ds = ISPRSDataset('data/processed', 'train', 'vaihingen')
val_ds   = ISPRSDataset('data/processed', 'val',   'vaihingen')
test_ds  = ISPRSDataset('data/processed', 'test',  'vaihingen')
print(f'\nТайлов: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

s = train_ds[0]
print(f'Image: {s["image"].shape} {s["image"].dtype}')
print(f'Mask:  {s["mask"].shape}  values={sorted(s["mask"].unique().tolist())}')

In [ ]:
# Ячейка 7: Обучение всех моделей
# Время: ~1-2 часа на GPU на каждую модель
import subprocess, json, os, time, sys

env = {**os.environ, 'PYTHONPATH': DIPLOMA_DIR}
EPOCHS     = 100
BATCH_SIZE = 4  # уменьши до 2 если мало памяти

# (model_name, learning_rate)
MODELS = [
    ('fcn', 1e-4), ('unet', 1e-4), ('deeplab', 1e-4),
    ('attention', 1e-4), ('segformer', 1e-4),
    ('unet_pt', 5e-5), ('attention_pt', 5e-5),
]

all_results = {}

for model_name, lr in MODELS:
    save_dir = f'experiments/{model_name}'

    if os.path.exists(f'{save_dir}/results.json'):
        res = json.load(open(f'{save_dir}/results.json'))
        print(f'{model_name}: уже обучена mIoU={res.get("best_miou",0):.4f}')
        all_results[model_name] = res
        continue

    print(f"\n{'='*50}  {model_name.upper()}  lr={lr}")
    os.makedirs(save_dir, exist_ok=True)
    t0 = time.time()

    result = subprocess.run(
        [sys.executable, 'src/training/train.py',
         '--model', model_name, '--dataset', 'vaihingen',
         '--dataset_path', 'data/processed',
         '--epochs', str(EPOCHS), '--batch_size', str(BATCH_SIZE),
         '--lr', str(lr), '--save_dir', save_dir, '--device', DEVICE],
        text=True, cwd=DIPLOMA_DIR, env=env
    )
    elapsed = time.time() - t0

    if result.stdout:
        for line in result.stdout.strip().split('\n')[-5:]:
            print(line)
    if result.returncode != 0:
        print('ОШИБКА:', result.stderr[-300:])
        continue

    if os.path.exists(f'{save_dir}/results.json'):
        res = json.load(open(f'{save_dir}/results.json'))
        all_results[model_name] = res
        print(f'mIoU={res.get("best_miou",0):.4f} ({elapsed/60:.0f} мин)')

print(f'\nГотово: {len(all_results)}/7 моделей')

In [ ]:
# Ячейка 8: Оценка на тестовой выборке
import torch, json, os
from src.models import get_model
from src.data.dataset import get_dataloader, get_dataset_info
from src.evaluation.metrics import MetricsTracker, compute_flops_params

device      = torch.device(DEVICE)
info        = get_dataset_info('vaihingen')
num_classes = info['num_classes']
class_names = info['class_names']
test_loader = get_dataloader('vaihingen', 'data/processed', 'test', batch_size=2, num_workers=0)

all_metrics = {}
for name in ['fcn','unet','deeplab','attention','segformer','unet_pt','attention_pt']:
    ckpt_path = f'experiments/{name}/best.pth'
    if not os.path.exists(ckpt_path):
        print(f'{name}: нет чекпоинта')
        continue

    model = get_model(name, num_classes=num_classes).to(device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt.get('model_state', ckpt), strict=True)
    model.eval()

    tracker = MetricsTracker(num_classes, class_names)
    with torch.no_grad():
        for batch in test_loader:
            preds = model(batch['image'].to(device)).argmax(1)
            tracker.update(preds.cpu(), batch['mask'])

    m  = tracker.compute()
    fp = compute_flops_params(model.cpu(), input_size=(1,3,512,512))
    all_metrics[name] = {'miou': m['miou'], 'boundary_iou': m['boundary_iou'],
                         'params_M': fp['params_M'], 'flops_G': fp['flops_G']}
    print(f"{name:<14} mIoU={m['miou']:.4f}  BIoU={m['boundary_iou']:.4f}  "
          f"{fp['params_M']:.1f}M  {fp['flops_G']:.1f}G")

os.makedirs('experiments/comparison', exist_ok=True)
with open('experiments/comparison/compare_results.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print('\nСохранено: experiments/comparison/compare_results.json')

In [ ]:
# Ячейка 9: Финальная таблица и проверка гипотез
import json
results = json.load(open('experiments/comparison/compare_results.json'))

print(f'{"Модель":<16} {"mIoU":>8} {"BIoU":>8} {"Params_M":>10} {"FLOPs_G":>10}')
print('─' * 56)
for name, m in sorted(results.items(), key=lambda x: -x[1]['miou']):
    print(f"{name:<16} {m['miou']:>8.4f} {m['boundary_iou']:>8.4f} "
          f"{m['params_M']:>10.1f} {m['flops_G']:>10.1f}")

print('\n=== Проверка гипотез (Attention vs U-Net) ===')
for label, base_key, att_key in [('scratch', 'unet', 'attention'),
                                  ('pretrained', 'unet_pt', 'attention_pt')]:
    if base_key not in results or att_key not in results:
        continue
    b, a = results[base_key], results[att_key]
    dm = a['miou'] - b['miou']
    rf = a['flops_G'] / b['flops_G']
    db = a['boundary_iou'] - b['boundary_iou']
    print(f'\n{label}:')
    print(f'  H1 mIoU +5pp:   {"V" if dm>=0.05 else "X"} {dm:+.4f}')
    print(f'  H2 FLOPs 120%:  {"V" if rf<=1.20 else "X"} {rf:.2f}x')
    print(f'  H3 BIoU +15pp:  {"V" if db>=0.15 else "X"} {db:+.4f}')

In [ ]:
# Ячейка 10: Графики кривых обучения
import json, os, matplotlib.pyplot as plt

colors = {'fcn':'#378ADD','unet':'#1D9E75','deeplab':'#EF9F27',
          'attention':'#E24B4A','segformer':'#534AB7',
          'unet_pt':'#085041','attention_pt':'#A32D2D'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ISPRS Vaihingen — кривые обучения', fontsize=13, fontweight='bold')

for name, color in colors.items():
    path = f'experiments/{name}/results.json'
    if not os.path.exists(path): continue
    h = json.load(open(path)).get('history', [])
    if not h: continue
    ls = '--' if '_pt' in name else '-'
    ep = [e['epoch'] for e in h]
    axes[0].plot(ep, [e['val_miou'] for e in h], label=name, color=color, ls=ls)
    axes[1].plot(ep, [e['val_loss'] for e in h], color=color, ls=ls)

for ax, t, y in zip(axes, ['Val mIoU','Val Loss'], ['mIoU','Loss']):
    ax.set(title=t, xlabel='Эпоха', ylabel=y)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig('experiments/comparison/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Ячейка 11: Инференс на тестовом тайле
import subprocess, sys, os, matplotlib.pyplot as plt, matplotlib.image as mpimg

env = {**os.environ, 'PYTHONPATH': DIPLOMA_DIR}
test_img = sorted(os.listdir('data/processed/vaihingen/test/images'))[0]
sample   = f'data/processed/vaihingen/test/images/{test_img}'
output   = 'output/sample_prediction.png'

os.makedirs('output', exist_ok=True)
result = subprocess.run(
    [sys.executable, 'src/inference.py',
     '--model', 'unet_pt',
     '--checkpoint', 'experiments/unet_pt/best.pth',
     '--image', sample, '--output', output, '--device', DEVICE],
    text=True, cwd=DIPLOMA_DIR, env=env
)
print(result.stdout)
if result.returncode != 0:
    print('ОШИБКА:', result.stderr[-300:])
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(mpimg.imread(sample))
    axes[0].set(title='Входное изображение', axis='off')
    axes[1].imshow(mpimg.imread(output))
    axes[1].set(title='Предсказание U-Net PT', axis='off')
    plt.tight_layout()
    plt.show()